# EU Market Pulse — Inflation Forecasting

## Objective

This notebook builds a simple linear trend forecast for inflation rate per 
country, projecting 2026 and 2027 based on the 2014-2025 historical trend. 

The goal is not predictive precision — inflation is influenced by many 
external factors a simple trend model cannot capture — but to provide a 
directional signal: **is inflation in each country trending toward greater 
stability, or greater volatility?** This complements the historical 
`inflation_stability_score` already calculated, adding a forward-looking view.

## Method

A simple linear regression (`scikit-learn`) is fit per country, using year as 
the predictor and inflation rate as the target. This method is transparent 
and easy to justify — the slope of the line directly represents the average 
yearly change in inflation, which is intuitive to explain to a business 
audience.

## 1. Load Data from Supabase

We load `market_indicators` directly from the database, reusing the 
connection already validated in the pipeline automation notebook.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv('../.env')
engine = create_engine(os.getenv('DATABASE_URL'))

df = pd.read_sql('SELECT country_code, year, inflation_rate FROM market_indicators ORDER BY country_code, year;', engine)

print(f"Shape: {df.shape}")
df.head()

Shape: (222, 3)


,country_code,year,inflation_rate
0,AT,2014,1.5
1,AT,2015,0.8
2,AT,2016,1.0
3,AT,2017,2.2
4,AT,2018,2.1


## 2. Fit Linear Trend per Country

For each country, we fit a simple linear regression using `year` as the 
predictor and `inflation_rate` as the target. The slope represents the 
average yearly change in inflation over the 2014-2025 period — a positive 
slope indicates rising/accelerating inflation over time, a negative slope 
indicates a cooling trend.

In [2]:
from sklearn.linear_model import LinearRegression
import numpy as np

results = []

for country in df['country_code'].unique():
    country_data = df[df['country_code'] == country].sort_values('year')
    
    X = country_data[['year']].values
    y = country_data['inflation_rate'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    slope = model.coef_[0]
    
    # Forecast 2026 and 2027
    forecast_2026 = model.predict([[2026]])[0]
    forecast_2027 = model.predict([[2027]])[0]
    
    results.append({
        'country_code': country,
        'trend_slope': round(slope, 3),
        'forecast_2026': round(forecast_2026, 2),
        'forecast_2027': round(forecast_2027, 2)
    })

forecast_df = pd.DataFrame(results).sort_values('trend_slope')
forecast_df

,country_code,trend_slope,forecast_2026,forecast_2027
2,CH,0.160,1.58,1.75
7,FI,0.275,3.56,3.84
8,FR,0.292,3.82,4.11
9,GB,0.314,4.52,4.83
5,DK,0.318,3.76,4.08
18,SE,0.363,4.83,5.19
13,IT,0.364,4.25,4.61
1,BE,0.373,5.11,5.48
4,DE,0.384,5.00,5.38
16,PT,0.386,4.43,4.82


## 2. Fit Linear Trend per Country (Recent Period: 2022-2025)

Fitting the regression on the full 2014-2025 period produces a misleading 
upward trend for every country, since it averages the pre-2022 low-inflation 
years with the 2022 energy-crisis spike — masking the cooling trend already 
visible in 2023-2025.

To capture the current trajectory more accurately, we restrict the model to 
**2022-2025** — starting at the inflation peak itself, through the most 
recent data. This is a deliberate trade-off: fewer data points reduce 
statistical robustness, but better reflect the present economic reality than 
the full decade.

In [3]:
from sklearn.linear_model import LinearRegression

RECENT_YEARS_START = 2022

df_recent = df[df['year'] >= RECENT_YEARS_START]

results = []

for country in df_recent['country_code'].unique():
    country_data = df_recent[df_recent['country_code'] == country].sort_values('year')
    
    X = country_data[['year']].values
    y = country_data['inflation_rate'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    slope = model.coef_[0]
    
    forecast_2026 = model.predict([[2026]])[0]
    forecast_2027 = model.predict([[2027]])[0]
    
    results.append({
        'country_code': country,
        'trend_slope': round(slope, 3),
        'forecast_2026': round(forecast_2026, 2),
        'forecast_2027': round(forecast_2027, 2)
    })

forecast_df = pd.DataFrame(results).sort_values('trend_slope')
forecast_df

,country_code,trend_slope,forecast_2026,forecast_2027
3,CZ,-4.68,-3.75,-8.43
10,HU,-4.60,-1.40,-6.00
14,PL,-3.68,-1.45,-5.13
13,NL,-2.67,-1.20,-3.87
12,IT,-2.61,-2.20,-4.81
4,DE,-2.27,-0.80,-3.07
5,DK,-2.25,-1.85,-4.10
11,IE,-2.19,-1.30,-3.49
9,GR,-2.04,-0.25,-2.29
17,SE,-2.04,-0.45,-2.49


## 2. Illustrative 1-Year Trend Signal (2022-2025)

**Note on scope:** this section is intentionally limited and exploratory. 
The project's core business question is already fully answered by the 
KPI scoring (Block 2) and SQL analysis (Block 3). This forecast is not 
a pillar of the analysis — it is a small, honestly-scoped illustration of 
directional trend, included to demonstrate awareness of when a simple model 
is (and isn't) appropriate.

A linear regression fit on 2022-2025 data was tested for 2-year-ahead 
forecasting, but produced economically implausible results (e.g. projected 
deflation below -8% by 2027) — an artifact of extrapolating a short, steep 
post-peak recovery slope too far into the future. 

To keep the signal honest, we:
- Forecast only **one year ahead (2026)**, not two
- Apply a **floor of 0%** to avoid implausible negative projections

This should be read as a rough directional signal ("this country's inflation 
is still cooling" vs. "already near stable"), not a precise prediction.

In [4]:
from sklearn.linear_model import LinearRegression

RECENT_YEARS_START = 2022
df_recent = df[df['year'] >= RECENT_YEARS_START]

results = []

for country in df_recent['country_code'].unique():
    country_data = df_recent[df_recent['country_code'] == country].sort_values('year')
    
    X = country_data[['year']].values
    y = country_data['inflation_rate'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    forecast_2026 = model.predict([[2026]])[0]
    forecast_2026_floored = max(forecast_2026, 0)  # sanity floor: no implausible negative inflation
    
    results.append({
        'country_code': country,
        'trend_slope': round(model.coef_[0], 2),
        'forecast_2026_raw': round(forecast_2026, 2),
        'forecast_2026': round(forecast_2026_floored, 2)
    })

forecast_df = pd.DataFrame(results).sort_values('forecast_2026')
forecast_df

,country_code,trend_slope,forecast_2026_raw,forecast_2026
1,BE,-1.99,0.00,0.00
2,CH,-0.90,-0.70,0.00
3,CZ,-4.68,-3.75,0.00
4,DE,-2.27,-0.80,0.00
6,ES,-1.73,0.00,0.00
5,DK,-2.25,-1.85,0.00
7,FI,-1.95,-1.30,0.00
8,FR,-1.84,-0.90,0.00
12,IT,-2.61,-2.20,0.00
9,GR,-2.04,-0.25,0.00


# EU Market Pulse — Forecasting Exploration (Not Included in Final Analysis)

## Summary

This notebook explored whether a simple linear trend model could add a 
forward-looking signal to the project's inflation analysis, complementing 
the historical `inflation_stability_score` (Block 2) and SQL trend queries 
(Block 3).

## What Was Tested

1. **Full period (2014-2025):** produced a misleading upward trend for every 
   country, since it averages pre-2022 low-inflation years with the 2022 
   energy-crisis spike — masking the cooling trend already visible in 
   2023-2025.
2. **Recent period (2022-2025), 2-year forecast:** produced economically 
   implausible results (e.g. projected deflation below -8% by 2027), an 
   artifact of extrapolating a short, steep post-peak recovery slope too far 
   into the future.
3. **Recent period (2022-2025), 1-year forecast with a 0% floor:** avoided 
   implausible negative values, but collapsed 16 of 18 countries to the same 
   0.00 floor value — eliminating the differentiation needed for the signal 
   to be useful.

## Decision: Forecasting Excluded from Final Analysis

With only 4 data points per country (2022-2025) covering an unusually sharp 
post-shock recovery period, a simple linear model does not have enough 
signal to produce a reliable or differentiated forecast. Rather than force 
a fragile model into the final deliverable, this analysis was deliberately 
excluded.

**The project's core business question — where a remote-first company should 
expand based on cost of living and inflation trends — remains fully answered 
by the historical KPI scoring (Block 2) and SQL analysis (Block 3)**, which 
do not depend on forward projection. 

Recognizing when a model is not fit for purpose, and choosing not to publish 
it, is treated here as the correct analytical outcome rather than a project 
gap.